# 微信聊天风格 LoRA 微调
## 基座: Qwen2.5-3B-Instruct | 方法: QLoRA | 环境: Google Colab 免费 T4

### 使用步骤
1. 点击菜单栏「代码执行程序」→「更改运行时类型」→ 选 T4 GPU → 保存
2. 把 lora_data/ 文件夹上传到你的 Google Drive 根目录
3. 从上到下逐个运行每个代码块（点击 ▶ 按钮或按 Ctrl+Enter）

## Step 1: 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 验证数据文件存在
import os
data_dir = '/content/drive/MyDrive/lora_data'
for f in ['train.json', 'valid.json', 'dataset_info.json']:
    path = os.path.join(data_dir, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f'✅ {f} ({size_mb:.1f} MB)')
    else:
        print(f'❌ {f} 未找到! 请确认 lora_data/ 已上传到 Google Drive 根目录')

## Step 2: 安装 LLaMA-Factory（约 3 分钟）

In [ ]:
# 克隆 LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git /content/LLaMA-Factory
!cd /content/LLaMA-Factory && pip install -e . -q
!pip install bitsandbytes transformers_stream_generator -q

print('✅ LLaMA-Factory 安装完成')

## Step 3: 上传数据到 Colab 本地

In [ ]:
import shutil, json, os

# 从 Google Drive 复制数据到 Colab 本地（快于直接读 Drive）
colab_data = '/content/data'
os.makedirs(colab_data, exist_ok=True)

drive_data = '/content/drive/MyDrive/lora_data'
for f in ['train.json', 'valid.json']:
    shutil.copy2(os.path.join(drive_data, f), os.path.join(colab_data, f))
    
# 注册数据集
factory_info_path = '/content/LLaMA-Factory/data/dataset_info.json'
with open(factory_info_path, 'r') as fp:
    info = json.load(fp)

info['chat_style_train'] = {
    'file_name': '/content/data/train.json',
    'formatting': 'sharegpt',
    'columns': {'messages': 'conversations'},
    'tags': {'role_tag': 'from', 'content_tag': 'value',
            'user_tag': 'human', 'assistant_tag': 'gpt', 'system_tag': 'system'}
}
info['chat_style_valid'] = {
    'file_name': '/content/data/valid.json',
    'formatting': 'sharegpt',
    'columns': {'messages': 'conversations'},
    'tags': {'role_tag': 'from', 'content_tag': 'value',
            'user_tag': 'human', 'assistant_tag': 'gpt', 'system_tag': 'system'}
}

with open(factory_info_path, 'w') as fp:
    json.dump(info, fp, ensure_ascii=False, indent=2)

# 验证
train = json.loads(open('/content/data/train.json').read())
valid = json.loads(open('/content/data/valid.json').read())
print(f'✅ 训练集: {len(train)} 样本 | 验证集: {len(valid)} 样本')
print(f'✅ 第一个样本预览:')
print(json.dumps(train[0], ensure_ascii=False, indent=2)[:300])

## Step 4: 启动训练（Qwen2.5-3B + LoRA）

⏱️ 预计 30-40 分钟 | 💾 显存 ~10GB (T4 16GB 够用，无需量化)

In [ ]:
!cd /content/LLaMA-Factory && llamafactory-cli train \
  --model_name_or_path Qwen/Qwen2.5-3B-Instruct \
  --dataset chat_style_train \
  --eval_dataset chat_style_valid \
  --output_dir /content/output/chat_style_lora \
  --stage sft \
  --do_train \
  --do_eval \
  --finetuning_type lora \
  --template qwen \
  --lora_rank 8 \
  --lora_target all \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 8 \
  --learning_rate 5e-5 \
  --num_train_epochs 2 \
  --lr_scheduler_type cosine \
  --warmup_ratio 0.1 \
  --cutoff_len 1024 \
  --logging_steps 50 \
  --save_steps 500 \
  --eval_steps 500 \
  --fp16 \
  --overwrite_output_dir

## Step 5: 打包下载 LoRA 权重

In [ ]:
# 打包
!cd /content/output && tar czf /content/chat_style_lora.tar.gz chat_style_lora/

# 复制到 Google Drive（方便下载）
import shutil
shutil.copy2('/content/chat_style_lora.tar.gz', '/content/drive/MyDrive/chat_style_lora.tar.gz')

# 查看文件大小
size = os.path.getsize('/content/chat_style_lora.tar.gz') / 1024 / 1024
print(f'✅ LoRA 权重已打包: {size:.1f} MB')
print(f'📍 Google Drive 路径: MyDrive/chat_style_lora.tar.gz')
print(f'📍 也可以从 Colab 左侧文件面板右键下载: chat_style_lora.tar.gz')

## (可选) Step 6: 快速测试效果

In [ ]:
# 加载 LoRA 试几条对话
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

model_path = 'Qwen/Qwen2.5-3B-Instruct'
lora_path = '/content/output/chat_style_lora'

print('加载模型...')
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_path, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True
)
model = PeftModel.from_pretrained(model, lora_path)

def chat(user_input):
    messages = [
        {'role': 'system', 'content': '你正在微信上与朋友聊天。'},
        {'role': 'user', 'content': user_input}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    return response.strip()

print('='*50)
print('试试输入几句话看效果:')
print('(输入 quit 退出)')
print('='*50)

while True:
    user = input('你: ')
    if user.lower() == 'quit':
        break
    resp = chat(user)
    print(f'AI: {resp}')
    print()